In [0]:


dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("source_table","")
# workspace,default,customers/products/orders
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
source_table = dbutils.widgets.get("source_table")

print(f"Loading data into {catalog}.{schema}")
print(f"Processing {source_table}")

#Create a reusable function to load data into Bronze
from pyspark.sql.functions import current_timestamp, lit

def load_to_bronze(source_table_name):
    
    df = spark.read.table(f"{catalog}.{schema}.{source_table_name}")

    bronze_df = (
        df
        .withColumn("ingestion_ts", current_timestamp())
        .withColumn("source_table", lit(source_table_name))
    )

    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        # .saveAsTable(f"workspace.default.bronze_{source_table_name}")
        .saveAsTable(f"{catalog}.{schema}.bronze_{source_table_name}")
    )

    print(f"Loaded bronze_{source_table_name}")

#Drop tables
# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_customers")
# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_products")
# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_orders")

load_to_bronze(source_table)

#display(spark.read.table("workspace.default.bronze_orders"))

